In [ ]:
import os
from google.colab import userdata


os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')


!kaggle datasets download -d ealaxi/paysim1 --force


!unzip paysim1.zip



In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.ensemble import IsolationForest

# NEW LINE in Cell 2
df = pd.read_csv('PS_20174392719_1491204439457_log.csv', nrows=10000)

def run_triple_engine_innovation(raw_data, sample_size=1000):
    print(f"🔄 Executing Triple-Engine Audit on {sample_size} rows...")
    df_audit = raw_data.iloc[:sample_size].copy()


    cols = ['Risk_A', 'Risk_B', 'Risk_C', 'Final_Score', 'drain_rate', 'user_history_avg', 'in_degree', 'is_ring']
    for col in cols: df_audit[col] = 0.0
    df_audit['Alert_Level'] = 'Green (Safe)'


    df_risk = df_audit[df_audit['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()


    history = df_risk.groupby('nameOrig')['amount'].transform('mean')
    df_risk['user_history_avg'] = history

    df_risk['deviation'] = df_risk['amount'] / (history + 1e-9)
    df_risk['drain_rate'] = (df_risk['amount'] / (df_risk['oldbalanceOrg'] + 1e-9)) * 100


    df_risk['Risk_A'] = np.clip((df_risk['deviation'] * 15) + (df_risk['drain_rate'] * 0.3), 0, 100)


    iso = IsolationForest(contamination=0.01, random_state=42)

    X = df_risk[['amount', 'oldbalanceOrg', 'newbalanceOrig', 'drain_rate']].fillna(0)
    iso.fit(X)
    ml_decision = -iso.decision_function(X)

    df_risk['Risk_B'] = ((ml_decision - ml_decision.min()) /
                         (ml_decision.max() - ml_decision.min() + 1e-9)) * 100


    G = nx.from_pandas_edgelist(df_risk, source='nameOrig', target='nameDest', create_using=nx.DiGraph())


    in_degrees = dict(G.in_degree())
    df_risk['in_degree'] = df_risk['nameDest'].map(in_degrees).fillna(0)



    df_risk['is_ring'] = df_risk.apply(lambda x: 1 if G.has_edge(x['nameDest'], x['nameOrig']) else 0, axis=1)


    df_risk['Risk_C'] = np.clip((df_risk['in_degree'] * 20) + (df_risk['is_ring'] * 50), 0, 100)

    df_risk['Final_Score'] = df_risk[['Risk_A', 'Risk_B', 'Risk_C']].max(axis=1)


    df_risk.loc[df_risk['Final_Score'] > 70, 'Alert_Level'] = 'Yellow (Review)'

    red_mask = (df_risk['Final_Score'] > 88) & (df_risk['amount'] > 150000) & (df_risk['drain_rate'] > 95)
    df_risk.loc[red_mask, 'Alert_Level'] = 'Red (Fraud)'

    df_audit.update(df_risk)
    return df_audit


df_final = run_triple_engine_innovation(df)


def generate_smart_reason(row):
    if row['Alert_Level'] == 'Green (Safe)': return "Normal behavior"
    reasons = []

    if row['Risk_C'] >= row['Final_Score'] and row['Risk_C'] > 50:
        reasons.append(f"GRAPH: {'Circular Ring' if row['is_ring'] else 'Mule Hub'} detected")
    if row['Risk_A'] >= row['Final_Score'] and row['Risk_A'] > 50:
        reasons.append("BEHAVIOR: Extreme liquidation relative to user history")
    if row['Risk_B'] >= row['Final_Score'] and row['Risk_B'] > 50:
        reasons.append("ML: Statistical pattern matches fraud signature")
    return " | ".join(reasons) if reasons else "High Anomaly Score"

df_final['Reason'] = df_final.apply(generate_smart_reason, axis=1)


In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.ensemble import IsolationForest

def run_triple_engine_innovation(raw_data):
    df_audit = raw_data.copy()

    cols = ['Risk_A', 'Risk_B', 'Risk_C', 'Final_Score', 'drain_rate', 'step_deviation', 'in_degree', 'is_zero_dest', 'is_ring']
    for col in cols:
        df_audit[col] = 0.0
    df_audit['Alert_Level'] = 'Green (Safe)'

    df_risk = df_audit[df_audit['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
    if df_risk.empty: return df_audit

    # ENGINE A: Behavioral Math
    step_avg = df_risk.groupby('step')['amount'].transform('mean')
    df_risk['step_deviation'] = df_risk['amount'] / (step_avg + 1e-9)
    df_risk['drain_rate'] = (df_risk['amount'] / (df_risk['oldbalanceOrg'] + 1e-9)) * 100
    df_risk['drain_rate'] = df_risk['drain_rate'].clip(0, 100)
    df_risk['Risk_A'] = np.clip((np.log1p(df_risk['step_deviation']) * 12) + (df_risk['drain_rate'] * 0.45), 0, 100)

    # ENGINE B: Isolation Forest (ML)
    iso = IsolationForest(contamination=0.02, random_state=42)
    features = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'drain_rate']
    X = df_risk[features].fillna(0)
    iso.fit(X)
    ml_scores = iso.decision_function(X)
    df_risk['Risk_B'] = 100 * (ml_scores.max() - ml_scores) / (ml_scores.max() - ml_scores.min() + 1e-9)
    df_risk['Risk_B'] = df_risk['Risk_B'].clip(0, 100)

    # ENGINE C: Graph Theory
    G = nx.from_pandas_edgelist(df_risk, source='nameOrig', target='nameDest', create_using=nx.DiGraph())
    in_degrees = dict(G.in_degree())
    df_risk['in_degree'] = df_risk['nameDest'].map(in_degrees).fillna(0)
    df_risk['is_ring'] = df_risk.apply(lambda x: 1 if G.has_edge(x['nameDest'], x['nameOrig']) else 0, axis=1)
    df_risk['is_zero_dest'] = (df_risk['oldbalanceDest'] == 0).astype(int)
    df_risk['Risk_C'] = np.clip((df_risk['is_ring'] * 50) + (df_risk['is_zero_dest'] * 35) + (np.log10(df_risk['in_degree'] + 1) * 20), 0, 100)

    # FINAL WEIGHTED SCORE
    df_risk['Final_Score'] = (df_risk['Risk_A'] * 0.4) + (df_risk['Risk_B'] * 0.3) + (df_risk['Risk_C'] * 0.3)

    max_engine = df_risk[['Risk_A', 'Risk_B', 'Risk_C']].max(axis=1)

    df_risk['Final_Score'] = np.where((max_engine >= 75) & (df_risk['Final_Score'] < 45), 46.0, df_risk['Final_Score'])

    df_risk['Final_Score'] = np.where((max_engine >= 60) & (df_risk['Final_Score'] < 35), 36.0, df_risk['Final_Score'])

    df_risk.loc[df_risk['Final_Score'] >= 38, 'Alert_Level'] = 'Yellow (Review)'
    df_risk.loc[df_risk['Final_Score'] >= 44, 'Alert_Level'] = 'Red (Fraud)'

    df_audit.update(df_risk)
    return df_audit

def generate_smart_reason(row):
    if row['Alert_Level'] == 'Green (Safe)': return "Consistent with historical patterns"
    reasons = []


    if row['Risk_C'] >= 60:
        reasons.append(f"GRAPH: {'Circular Flow Detected' if row['is_ring'] else 'Hidden Mule Hub Drop'}")
    if row['Risk_A'] >= 60:
        reasons.append("BEHAVIOR: Abnormal business volume & high drain rate")
    if row['Risk_B'] >= 60:
        reasons.append("ML: Statistical outlier in feature space")

    return " | ".join(reasons) if reasons else "Multi-vector anomaly detected"

df_final = run_triple_engine_innovation(df)
df_final['Reason'] = df_final.apply(generate_smart_reason, axis=1)

print("\n--- PROTOTYPE EFFICIENCY REPORT ---")
stats = df_final['Alert_Level'].value_counts()
print(stats)
fraud_percent = ( (stats.get('Red (Fraud)', 0) + stats.get('Yellow (Review)', 0)) / len(df_final) ) * 100
print(f"Total Flagged (Red + Yellow): {fraud_percent:.2f}%")

export_cols = ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
               'nameDest', 'oldbalanceDest', 'newbalanceDest',
               'drain_rate', 'Risk_A', 'Risk_B', 'Risk_C', 'Final_Score', 'Alert_Level', 'Reason']

df_final[export_cols].to_csv('dashboard_dataV2.csv', index=False)
print("\nSUCCESS: 'dashboard_dataV2.csv' is ready.")

files.download('dashboard_dataV2.csv')


--- PROTOTYPE EFFICIENCY REPORT ---
Alert_Level
Green (Safe)       9748
Yellow (Review)     138
Red (Fraud)         114
Name: count, dtype: int64
Total Flagged (Red + Yellow): 2.52%

SUCCESS: 'dashboard_dataV2.csv' is ready.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>